In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [2]:
train = pd.read_csv('data/kernel/Unit_commitment_decisions.csv')
train['starttime'] = pd.to_datetime(train['starttime'], utc=True).dt.tz_localize(None)
target_columns = train.columns[3:]

def read_time_table(path):
    table = pd.read_csv(path)
    table['date'] = pd.to_datetime(table['date'], utc=True).dt.tz_localize(None)
    return table.set_index('date')

price = read_time_table('data/kernel/Historical_day_ahead_price_2015_2025.csv')
inflow = read_time_table('data/kernel/Historical_inflow_1958_2025.csv')
volume = read_time_table('data/kernel/Historical_volume_2015_2024.csv')
water_value = read_time_table('data/kernel/Synthetic_water_value_2015_2024.csv')
min_volume = read_time_table('data/extended/Constraint_min_volume.csv')
min_flow = read_time_table('data/extended/Constraint_min_flow.csv')

In [3]:
def weekly_window(table, starts):
    timestamps = starts.to_numpy()[:, None] + np.arange(168, dtype='timedelta64[h]')
    values = table.reindex(timestamps.ravel()).to_numpy()
    return values.reshape(len(starts), 168, table.shape[1])

def make_features(cases):
    starts = cases['starttime']
    prices = weekly_window(price, starts)[:, :, 0]
    mean_price = prices.mean(axis=1, keepdims=True)
    spread = prices.std(axis=1, keepdims=True) + 1
    features = {
        'hour': np.tile(np.arange(168) % 24, len(cases)),
        'day': np.tile(np.arange(168) // 24, len(cases)),
        'month': np.repeat(starts.dt.month.to_numpy(), 168),
        'day_of_year': np.repeat(starts.dt.dayofyear.to_numpy(), 168),
        'relative_price': ((prices - mean_price) / spread).ravel(),
        'price_rank': pd.DataFrame(prices).rank(axis=1, pct=True).to_numpy().ravel(),
    }
    initial = volume.reindex(starts)
    for name in volume.columns.drop(['Hyljelihyl', 'Vatjern']):
        features[f'volume_{name}'] = np.repeat(initial[name].to_numpy(), 168)

    end_values = water_value.reindex(starts + pd.Timedelta(days=7))
    weekly = {}
    nearby = {}
    for name in end_values:
        value = end_values[name].to_numpy()[:, None]
        features[f'margin_water_end_{name}'] = ((prices - value) / spread).ravel()
        weekly[f'weekly_margin_{name}'] = np.repeat(((mean_price - value) / spread).ravel(), 168)
        weekly[f'profitable_share_{name}'] = np.repeat((prices > value).mean(axis=1), 168)
        profitable = pd.DataFrame(prices > value)
        nearby[f'local_profitable_{name}'] = profitable.T.rolling(12, center=True, min_periods=1).mean().T.to_numpy().ravel()

    for prefix, table in [('inflow', inflow), ('min_volume', min_volume), ('min_flow', min_flow)]:
        values = weekly_window(table, starts)
        for j, name in enumerate(table.columns):
            features[f'{prefix}_{name}'] = values[:, :, j].ravel()
    features.update(weekly)
    local_price = pd.DataFrame(prices).T.rolling(24, center=True, min_periods=1).mean().T.to_numpy()
    features['local_price'] = ((local_price - mean_price) / spread).ravel()
    features['price_rise'] = (np.diff(prices, axis=1, prepend=prices[:, :1]) / spread).ravel()
    features['next_price_change'] = (np.diff(prices, axis=1, append=prices[:, -1:]) / spread).ravel()
    features['horizon_hour'] = np.tile(np.arange(168), len(cases))
    features['relative_spread'] = np.repeat((spread / (np.abs(mean_price) + 1)).ravel(), 168)
    features.update(nearby)
    for name in volume:
        flow = weekly_window(inflow[[name]], starts)[:, :, 0]
        weekly_flow = flow.sum(axis=1) * 3600 / 1e6
        features[f'weekly_inflow_{name}'] = np.repeat(flow.mean(axis=1), 168)
        features[f'inflow_ratio_{name}'] = np.repeat(weekly_flow / (initial[name].to_numpy() + 0.1), 168)
    for name in min_volume:
        minimum = weekly_window(min_volume[[name]], starts)[:, :, 0]
        features[f'headroom_{name}'] = (initial[name].to_numpy()[:, None] - minimum).ravel()
    return pd.DataFrame(features, dtype=np.float32)


In [4]:
reservoirs = {
    'Hogga': ['Bandak'], 'Tokke': ['Vinjevatn', 'Vaamarvatn', 'Bandak'],
    'Lio': ['Byrtevatn', 'Botnedalsvatn', 'Bandak'], 'Byrte': ['Botnedalsvatn', 'Byrtevatn'],
    'Vinje': ['Vaamarvatn', 'Totak', 'Vinjevatn'], 'Haukeli': ['Vatjern', 'Langeidvatn'],
    'Songa': ['Songavatn', 'Bitdalsvatn', 'Vaamarvatn'],
    'Kjela': ['Foersvatn', 'Bordalsvatn', 'Venemo'],
    'Vesle Kjela': ['Kjelavatn', 'Foersvatn', 'Staavatn'],
}
plants = [
    'Hogga', 'Tokke', 'Tokke', 'Tokke', 'Tokke', 'Lio', 'Byrte',
    'Vinje', 'Vinje', 'Vinje', 'Haukeli', 'Songa', 'Kjela', 'Vesle Kjela',
]
test = pd.read_csv('prediction_mapping.csv', parse_dates=['starttime'])
X = make_features(train)
X_test = make_features(test)
y = train[target_columns].to_numpy(dtype=np.uint8).reshape(-1, 14, 168)
y = y.transpose(0, 2, 1).reshape(-1, 14)
core = [c for c in X if not c.startswith(('weekly_inflow_', 'inflow_ratio_', 'headroom_'))]
shared = [c for c in core if not c.startswith(('volume_', 'inflow_', 'margin_', 'weekly_margin_', 'profitable_share_', 'local_profitable_'))]
predictions = []

for j, plant in enumerate(plants):
    local = [c for c in core if c in shared or any(c.endswith('_' + name) for name in reservoirs[plant])]
    columns = core if plant in ('Hogga', 'Lio', 'Kjela') else local
    settings = dict(n_estimators=40, max_depth=20, min_samples_leaf=5, max_samples=0.25, max_features=0.7)
    if plant == 'Haukeli':
        columns = list(X.columns)
        settings = dict(n_estimators=60, min_samples_leaf=2, max_samples=0.35)
    elif plant == 'Vinje':
        settings = dict(n_estimators=100, max_samples=0.5)
    model = RandomForestClassifier(random_state=67, **settings)
    model.fit(X[columns], y[:, j])
    probabilities = model.predict_proba(X_test[columns])[:, 1]
    if plant == 'Lio':
        deeper = RandomForestClassifier(n_estimators=100, max_samples=0.5, random_state=67)
        deeper.fit(X[local], y[:, j])
        probabilities = (probabilities + deeper.predict_proba(X_test[local])[:, 1]) / 2
    predictions.append(probabilities)

wide = np.column_stack(predictions).reshape(-1, 168, 14).transpose(0, 2, 1).reshape(-1, len(target_columns))
submission = pd.DataFrame(wide, columns=target_columns)
submission.insert(0, 'Run No', test['Run No'].to_numpy())
submission.to_csv('submissions/model3short.csv', index=False)
